# Rancangan Sistem

Rancangan awal, sistem ini digunakan untuk menganalisis penjualan dengan periode bulanan dalam satu tahun.

# Preprocessing Data

In [ ]:
# Upload dataset
from google.colab import files
files.upload()

Saving Dataset_sales.csv to Dataset_sales (2).csv


{'Dataset_sales (2).csv': b'Transaction_ID,Date,Product_Name,Category,Units_Sold,Unit_Price,Total_Revenue,Store_Location,Payment_Method\nT0001,2024-10-06,Pepsodent Toothpaste,Personal Care,N/A,15000,675000.0,Jakarta,Cash\nT0002,2024-10-01,ABC Kecap Manis 620ml,Groceries,60,18000,1080000.0,Jakarta,Card\nT0003,2024-10-01,Lifebuoy Body Wash,Personal Care,17,25000,425000.0,Medan,Cash\nT0004,2024-10-06,Milo 1kg,Drinks,95,90000,8550000.0,Bandung,Cash\nT0005,2024-10-03,Indomie Goreng,Instant Noodles,30,3000,90000.0,Surabaya,Card\nT0006,2024-10-02,Sari Roti Tawar,Snacks,25,12000,300000.0,Jakarta,Card\nT0007,2024-10-02,SilverQueen Milk Chocolate,Snacks,43,12000,516000.0,Jakarta,Card\nT0008,2024-10-02,Nabati Wafer,Snacks,82,5000,410000.0,Bandung,Card\nT0009,2024-10-06,Rexona Deodorant,Personal Care,39,20000,780000.0,Jakarta,Cash\nT0010,2024-10-01,Aqua 600ml,Drinks,70,4000,280000.0,Bandung,Cash\nT0011,2024-10-06,Sampoerna Mild 16,Smokes,93,30000,2790000.0,Surabaya,Cash\nT0012,2024-10-06,Sari Roti

In [ ]:
# Membaca dataset
import pandas as pd
df = pd.read_csv("Dataset_sales.csv")

In [ ]:
#Melihat isi dataset
df.head()

,Transaction_ID,Date,Product_Name,Category,Units_Sold,Unit_Price,Total_Revenue,Store_Location,Payment_Method
0,T0001,2024-10-06,Pepsodent Toothpaste,Personal Care,NaN,15000,675000.0,Jakarta,Cash
1,T0002,2024-10-01,ABC Kecap Manis 620ml,Groceries,60.0,18000,1080000.0,Jakarta,Card
2,T0003,2024-10-01,Lifebuoy Body Wash,Personal Care,17.0,25000,425000.0,Medan,Cash
3,T0004,2024-10-06,Milo 1kg,Drinks,95.0,90000,8550000.0,Bandung,Cash
4,T0005,2024-10-03,Indomie Goreng,Instant Noodles,30.0,3000,90000.0,Surabaya,Card


In [ ]:
# Mengubah Unit_Price jadi angka
df['Unit_Price'] = pd.to_numeric(df['Unit_Price'], errors='coerce')

In [ ]:
# Menangani Missing Value
df['Units_Sold'] = df['Units_Sold'].fillna(df['Units_Sold'].median())
df['Unit_Price'] = df['Unit_Price'].fillna(df['Unit_Price'].median())

In [ ]:
# Mengisi Total_Revenue yang kosong
df.loc[df['Total_Revenue'].isna(), 'Total_Revenue'] = \
    df['Units_Sold'] * df['Unit_Price']

In [ ]:
# Mengubah tipe tanggal
df['Date'] = pd.to_datetime(df['Date'])

In [ ]:
# Cek struktur data
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Transaction_ID  4000 non-null   object        
 1   Date            4000 non-null   datetime64[ns]
 2   Product_Name    4000 non-null   object        
 3   Category        4000 non-null   object        
 4   Units_Sold      4000 non-null   float64       
 5   Unit_Price      4000 non-null   float64       
 6   Total_Revenue   4000 non-null   float64       
 7   Store_Location  4000 non-null   object        
 8   Payment_Method  4000 non-null   object        
dtypes: datetime64[ns](1), float64(3), object(5)
memory usage: 281.4+ KB


In [ ]:
# Cek data duplikat
df['Transaction_ID'].duplicated().sum()

np.int64(0)

In [ ]:
# Memecah tanggal jadi Tahun, Bulan, Hari
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day

In [ ]:
# Perhitungan revenue per toko
store_perf = df.groupby('Store_Location')['Total_Revenue'].sum().reset_index()

In [ ]:
# Agregasi Bulanan
df.groupby(['Year','Month','Store_Location'])['Total_Revenue'].sum()

Year  Month  Store_Location
2024  10     Bandung           1.091441e+09
             Jakarta           1.326678e+09
             Medan             1.106365e+09
             Surabaya          1.150999e+09
Name: Total_Revenue, dtype: float64

# Mendalami Dataset

In [ ]:
# Menampilkan tahun yang tersedia
tahun = df['Date'].dt.year.unique()
print("Tahun yang ada:", sorted(tahun))

Tahun yang ada: [np.int32(2024)]


In [ ]:
# Menampilkan bulan yang tersedia
bulan = df['Date'].dt.month.unique()
print("Bulan yang ada:", sorted(bulan))

Bulan yang ada: [np.int32(10)]


In [ ]:
# Menampilkan hari (tanggal) yang tersedia
hari = df['Date'].dt.day.unique()
print("Hari yang ada:", sorted(hari))

Hari yang ada: [np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6)]


In [ ]:
# Lokasi Toko
lokasi_toko = df['Store_Location'].unique()
print("Daftar Lokasi Toko:")
print(lokasi_toko)

Daftar Lokasi Toko:
['Jakarta' 'Medan' 'Bandung' 'Surabaya']


Karena dataset hanya berisi tanggal transaksi bulan 10, tahun 2024, tanggal 1-6. Kami merubah fitur sistem menjadi periode analisis penjualan menjadi harian.

# Fakta

In [ ]:
# Total revenue harian (SEMUA toko)
revenue_harian_global = df.groupby('Date')['Total_Revenue'].sum().reset_index()
revenue_harian_global = revenue_harian_global.sort_values('Date')
print(revenue_harian_global)

        Date  Total_Revenue
0 2024-10-01    761304000.0
1 2024-10-02    734294700.0
2 2024-10-03    774987500.0
3 2024-10-04    804222800.0
4 2024-10-05    764919600.0
5 2024-10-06    835753800.0


In [ ]:
# Total revenue harian PER TOKO
revenue_harian_toko = df.groupby(['Date','Store_Location'])['Total_Revenue'].sum().reset_index()
revenue_harian_toko = revenue_harian_toko.sort_values(['Store_Location','Date'])
print(revenue_harian_toko)

         Date Store_Location  Total_Revenue
0  2024-10-01        Bandung    190071000.0
4  2024-10-02        Bandung    154453500.0
8  2024-10-03        Bandung    183349400.0
12 2024-10-04        Bandung    152705200.0
16 2024-10-05        Bandung    170320800.0
20 2024-10-06        Bandung    240541000.0
1  2024-10-01        Jakarta    208921700.0
5  2024-10-02        Jakarta    191537200.0
9  2024-10-03        Jakarta    210854100.0
13 2024-10-04        Jakarta    260003500.0
17 2024-10-05        Jakarta    240865600.0
21 2024-10-06        Jakarta    214495400.0
2  2024-10-01          Medan    193123800.0
6  2024-10-02          Medan    160458100.0
10 2024-10-03          Medan    173878000.0
14 2024-10-04          Medan    218243000.0
18 2024-10-05          Medan    169226800.0
22 2024-10-06          Medan    191435700.0
3  2024-10-01       Surabaya    169187500.0
7  2024-10-02       Surabaya    227845900.0
11 2024-10-03       Surabaya    206906000.0
15 2024-10-04       Surabaya    

In [ ]:
# Perubahan hari ke hari
revenue_harian_global['Prev'] = revenue_harian_global['Total_Revenue'].shift(1)
revenue_harian_global['Perubahan'] = revenue_harian_global['Total_Revenue'] - revenue_harian_global['Prev']

In [ ]:
# Status Tren Harian Global Seluruh Toko
def status_tren(x):
    if pd.isna(x):
        return "Tidak ada data"
    elif x > 0:
        return "Naik"
    elif x < 0:
        return "Turun"
    else:
        return "Stagnan"

revenue_harian_global['Status'] = revenue_harian_global['Perubahan'].apply(status_tren)
print(revenue_harian_global)

        Date  Total_Revenue         Prev   Perubahan          Status
0 2024-10-01    761304000.0          NaN         NaN  Tidak ada data
1 2024-10-02    734294700.0  761304000.0 -27009300.0           Turun
2 2024-10-03    774987500.0  734294700.0  40692800.0            Naik
3 2024-10-04    804222800.0  774987500.0  29235300.0            Naik
4 2024-10-05    764919600.0  804222800.0 -39303200.0           Turun
5 2024-10-06    835753800.0  764919600.0  70834200.0            Naik


In [ ]:
# Status tren (naik / turun / stagnan)
def status_tren(x):
    if pd.isna(x):
        return "Tidak ada data"
    elif x > 0:
        return "Naik"
    elif x < 0:
        return "Turun"
    else:
        return "Stagnan"

revenue_harian_global['Status'] = revenue_harian_global['Perubahan'].apply(status_tren)

print(revenue_harian_global)

        Date  Total_Revenue         Prev   Perubahan          Status
0 2024-10-01    761304000.0          NaN         NaN  Tidak ada data
1 2024-10-02    734294700.0  761304000.0 -27009300.0           Turun
2 2024-10-03    774987500.0  734294700.0  40692800.0            Naik
3 2024-10-04    804222800.0  774987500.0  29235300.0            Naik
4 2024-10-05    764919600.0  804222800.0 -39303200.0           Turun
5 2024-10-06    835753800.0  764919600.0  70834200.0            Naik


In [ ]:
# Hari penjualan terbaik & terburuk
hari_terbaik = revenue_harian_global.loc[revenue_harian_global['Total_Revenue'].idxmax()]
hari_terburuk = revenue_harian_global.loc[revenue_harian_global['Total_Revenue'].idxmin()]

print("Hari terbaik:\n", hari_terbaik)
print("Hari terburuk:\n", hari_terburuk)

Hari terbaik:
 Date             2024-10-06 00:00:00
Total_Revenue            835753800.0
Prev                     764919600.0
Perubahan                 70834200.0
Status                          Naik
Name: 5, dtype: object
Hari terburuk:
 Date             2024-10-02 00:00:00
Total_Revenue            734294700.0
Prev                     761304000.0
Perubahan                -27009300.0
Status                         Turun
Name: 1, dtype: object


In [ ]:
# Kontribusi penjualan tiap toko (%)
total = df['Total_Revenue'].sum()
kontribusi = df.groupby('Store_Location')['Total_Revenue'].sum().reset_index()
kontribusi['Persentase'] = (kontribusi['Total_Revenue'] / total) * 100
print(kontribusi)

  Store_Location  Total_Revenue  Persentase
0        Bandung   1.091441e+09   23.343921
1        Jakarta   1.326678e+09   28.375200
2          Medan   1.106365e+09   23.663128
3       Surabaya   1.150999e+09   24.617751


In [ ]:
# Tren harian PER TOKO (status)
revenue_harian_toko['Prev'] = revenue_harian_toko.groupby('Store_Location')['Total_Revenue'].shift(1)
revenue_harian_toko['Perubahan'] = revenue_harian_toko['Total_Revenue'] - revenue_harian_toko['Prev']
revenue_harian_toko['Status'] = revenue_harian_toko['Perubahan'].apply(status_tren)
output_toko = revenue_harian_toko[['Date', 'Store_Location', 'Total_Revenue', 'Status']]
output_toko = output_toko.sort_values(['Store_Location', 'Date'])
print(output_toko)

         Date Store_Location  Total_Revenue          Status
0  2024-10-01        Bandung    190071000.0  Tidak ada data
4  2024-10-02        Bandung    154453500.0           Turun
8  2024-10-03        Bandung    183349400.0            Naik
12 2024-10-04        Bandung    152705200.0           Turun
16 2024-10-05        Bandung    170320800.0            Naik
20 2024-10-06        Bandung    240541000.0            Naik
1  2024-10-01        Jakarta    208921700.0  Tidak ada data
5  2024-10-02        Jakarta    191537200.0           Turun
9  2024-10-03        Jakarta    210854100.0            Naik
13 2024-10-04        Jakarta    260003500.0            Naik
17 2024-10-05        Jakarta    240865600.0           Turun
21 2024-10-06        Jakarta    214495400.0           Turun
2  2024-10-01          Medan    193123800.0  Tidak ada data
6  2024-10-02          Medan    160458100.0           Turun
10 2024-10-03          Medan    173878000.0            Naik
14 2024-10-04          Medan    21824300

# Backward Chaining

Goal dan Rule

*   Apakah tren penjualan harian global naik?

R1:
IF total_penjualan_hari_terakhir > total_penjualan_hari_sebelumnya
THEN tren_penjualan_global = naik

R2:
IF total_penjualan_hari_terakhir < total_penjualan_hari_sebelumnya
THEN tren_penjualan_global = turun

R3:
IF total_penjualan_hari_terakhir = total_penjualan_hari_sebelumnya
THEN tren_penjualan_global = stagnan

*   Apakah tanggal 3 adalah hari dengan penjualan tertinggi?

R4:
IF total_penjualan_tanggal_3 = nilai maksimum dari seluruh tanggal
THEN tanggal_3 = hari_penjualan_tertinggi

*   Apakah Jakarta adalah toko penjualan terendah?

R5:
IF total_penjualan_Jakarta = nilai minimum dari seluruh toko
THEN Jakarta = toko_penjualan_terendah

# Implementasi rule ke Python

In [ ]:
# R1-R3
def cek_tren_global(revenue_harian_global):
    last = revenue_harian_global.iloc[-1]['Total_Revenue']
    prev = revenue_harian_global.iloc[-2]['Total_Revenue']

    if last > prev:
        return "Naik"
    elif last < prev:
        return "Turun"
    else:
        return "Stagnan"

In [ ]:
# R4 (tanggal 3 tertinggi)
def cek_tanggal_tertinggi(revenue_harian_global, tanggal):
    max_val = revenue_harian_global['Total_Revenue'].max()

    nilai_tanggal = revenue_harian_global[
        revenue_harian_global['Date'].dt.day == tanggal
    ]['Total_Revenue'].values[0]

    return nilai_tanggal == max_val

In [ ]:
# R5 (Jakarta terendah)
def cek_toko_terendah(kontribusi_toko):
    min_val = kontribusi_toko['Total_Revenue'].min()

    jakarta = kontribusi_toko[
        kontribusi_toko['Store_Location'] == 'Jakarta'
    ]['Total_Revenue'].values[0]

    return jakarta == min_val

In [ ]:
def backward_chaining(goal, data):

    if goal == "tren_global_naik":
        last = data.iloc[-1]['Total_Revenue']
        prev = data.iloc[-2]['Total_Revenue']
        return last > prev

    elif goal == "tanggal_3_tertinggi":
        max_val = data['Total_Revenue'].max()
        val = data[data['Date'].dt.day == 3]['Total_Revenue'].values[0]
        return val == max_val

    else:
        return "Goal tidak dikenali"

In [ ]:
# Pengujian
print("Apakah tren global naik?")
print(cek_tren_global(revenue_harian_global))

print("\nApakah tanggal 3 tertinggi?")
print(cek_tanggal_tertinggi(revenue_harian_global, 3))

print("\nApakah Jakarta toko terendah?")
print(cek_toko_terendah(store_perf))

Apakah tren global naik?
Naik

Apakah tanggal 3 tertinggi?
False

Apakah Jakarta toko terendah?
False
